# FLUX.2 Klein Image Generation — Google Colab

This notebook runs the same Flask web UI as `python app.py`, but inside
Google Colab on a GPU runtime. Steps:

1. **Runtime → Change runtime type → GPU** (an A100 / L4 / T4 with enough VRAM is recommended; the 4B model needs ~16 GB).
2. Run the cells in order. The last cell prints a clickable URL to the web UI.

You will need a Hugging Face token with access to
[FLUX.2-klein-4B](https://huggingface.co/black-forest-labs/FLUX.2-klein-4B) /
[FLUX.2-klein-9B](https://huggingface.co/black-forest-labs/FLUX.2-klein-9B)
(accept the license on each model page first).

## 1. Clone the repository

In [ ]:
import os, pathlib

REPO_URL = "https://github.com/Stx666Michael/image-gen-edit.git"
REPO_DIR = "/content/image-gen-edit"

if not pathlib.Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

## 2. Install dependencies

Colab already ships with a CUDA build of PyTorch, so we install everything
from `requirements.txt` *except* the torch packages (which would otherwise
be replaced with the CPU build).

In [ ]:
!grep -viE '^(torch|torchvision|torchaudio)([<>=!~ ]|$)' requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt

## 3. Log in to Hugging Face

Paste a token from https://huggingface.co/settings/tokens when prompted.

In [ ]:
from huggingface_hub import login
login()

## 4. Sanity check the GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM free: {free/1e9:.1f} / {total/1e9:.1f} GB")

## 5. Start the Flask web UI

The server runs in a background thread on port `5000` and is exposed
through Colab's built-in port proxy — click the printed URL to open the UI.

The first generation will be slow because the model weights are downloaded
and loaded onto the GPU on demand.

In [ ]:
import threading, time
from app import app

PORT = 5000

def _run():
    # threaded=True so progress polling doesn't block generation requests.
    app.run(host="0.0.0.0", port=PORT, debug=False, use_reloader=False, threaded=True)

if not any(t.name == "flux-web" for t in threading.enumerate()):
    threading.Thread(target=_run, name="flux-web", daemon=True).start()
    time.sleep(2)

from google.colab.output import serve_kernel_port_as_window, serve_kernel_port_as_iframe
print("Open the web UI in a new tab:")
serve_kernel_port_as_window(PORT)
print("Or use the embedded view below:")
serve_kernel_port_as_iframe(PORT, height="800")

## 6. (Optional) Tail the server log

Run this cell if you want to see request logs / tracebacks while the UI is running.

In [ ]:
import logging, sys
logging.basicConfig(stream=sys.stdout, level=logging.INFO, force=True)
logging.getLogger("werkzeug").setLevel(logging.INFO)